In [ ]:
import os
import time
import csv
import psutil
import subprocess
import ast
import random
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai
import pandas as pd

In [ ]:

# -----------------------------------------------------------
#         EXPERIMENT & SAMPLING CONFIGURATION
# -----------------------------------------------------------
MODELS = [
    "gemini-2.0-flash",
    # "gemini-2.5-pro-exp-03-25",
    # "gemini-2.0-flash-thinking-exp-01-21",
]
# Definimos dos conjuntos de parámetros, cada uno con una temperatura fija:
# - Determinista: temperature=0 con {"top_p": 0.1, "top_k": 10}
# - Menos determinista: temperature=1 con {"top_p": 0.95, "top_k": 100}
PARAM_SETS = [
    {"temperature": 0, "top_p": 0.1, "top_k": 10},    # Determinista
    {"temperature": 1, "top_p": 0.95, "top_k": 100},   # Menos determinista
]

# Tamaño total de la muestra estratificada (se quieren 113 problemas en total)
TOTAL_SAMPLE_SIZE = 113

# Variabilidad (S_h) por estrato de dificultad.
VARIABILITY = {
    "Fácil": 1.0,    # Fácil
    "Mediano": 2.0,  # Mediano
    "Difícil": 3.0   # Difícil
}

# Tiempo máximo de ejecución para cada script generado (en segundos)
TIMEOUT_EXEC = 300  # 5 minutos

# -----------------------------------------------------------
#         FILE PATH CONFIGURATION
# -----------------------------------------------------------
INPUT_FILENAME = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\data\raw\leetcode_problems.csv"
BASE_OUTPUT_DIR = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs1134"
BASE_RESULTS_DIR = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\results1134"
SAMPLE_ALLOCATION_CSV = "sample_allocation1.csv"
CHECKPOINT_CSV = "results_checkpoint1.csv"

# -----------------------------------------------------------
#              API KEYS MANAGEMENT
# -----------------------------------------------------------
API_KEYS = []
CURRENT_KEY_INDEX = 0

def setup_environment():
    """
    Carga las variables de entorno y configura la API de Gemini con la primera key disponible.
    """
    load_dotenv(override=True)
    keys = []
    for i in range(1, 5):
        key = os.environ.get(f"GEMINI_API_KEY_{i}")
        if key:
            keys.append(key)
    if not keys:
        raise EnvironmentError("No se encontraron API keys en el archivo .env.")
    global API_KEYS, CURRENT_KEY_INDEX
    API_KEYS = keys
    CURRENT_KEY_INDEX = 0
    current_key = API_KEYS[CURRENT_KEY_INDEX]
    print(f"Using API key: {current_key[:1]}...{current_key[-3:]}")
    genai.configure(api_key=current_key)

def rotate_api_key():
    """
    Rota a la siguiente API key disponible y configura la API de Gemini.
    """
    global CURRENT_KEY_INDEX, API_KEYS
    CURRENT_KEY_INDEX = (CURRENT_KEY_INDEX + 1) % len(API_KEYS)
    new_key = API_KEYS[CURRENT_KEY_INDEX]
    print(f"Rotating API key. New key: {new_key[:1]}...{new_key[-3:]}")
    genai.configure(api_key=new_key)

# -----------------------------------------------------------
#         STRATIFIED SAMPLING FUNCTION (Neyman Allocation)
# -----------------------------------------------------------
def stratified_sampling(input_csv: str, total_sample_size: int, variability: dict):
    """
    Realiza un muestreo estratificado usando asignación de Neyman a partir de un CSV que contiene una columna 'Difficulty'.
    Devuelve un DataFrame con la muestra seleccionada y guarda los detalles de asignación en SAMPLE_ALLOCATION_CSV.
    """
    import pandas as pd
    df = pd.read_csv(input_csv, encoding="utf-8")
    # Agrupamos por 'Difficulty' (valores esperados: 'Fácil', 'Mediano', 'Difícil')
    strata = df.groupby("Difficulty")
    allocation = {}
    total_weight = 0
    for name, group in strata:
        N_h = len(group)
        S_h = variability.get(name, 1.0)
        allocation[name] = {"N_h": N_h, "S_h": S_h}
        total_weight += N_h * S_h

    sample_indices = []
    allocation_details = []
    for name, info in allocation.items():
        n_h = int(round(total_sample_size * (info["N_h"] * info["S_h"]) / total_weight))
        n_h = min(n_h, info["N_h"])
        allocation[name]["n_h"] = n_h
        group_indices = df[df["Difficulty"] == name].index.tolist()
        selected = random.sample(group_indices, n_h)
        sample_indices.extend(selected)
        allocation_details.append({
            "Difficulty": name,
            "N_h": info["N_h"],
            "S_h": info["S_h"],
            "n_h": n_h
        })

    alloc_df = pd.DataFrame(allocation_details)
    alloc_df.to_csv(SAMPLE_ALLOCATION_CSV, index=False, encoding="utf-8")
    print(f"Sample allocation saved in {SAMPLE_ALLOCATION_CSV}")
    sample_df = df.loc[sample_indices].copy()
    return sample_df

# -----------------------------------------------------------
#         UTILITY FUNCTIONS
# -----------------------------------------------------------
def extract_code(response_text: str) -> str:
    """
    Extrae el código Python del texto de respuesta eliminando los delimitadores de Markdown.
    """
    if "```python" in response_text:
        parts = response_text.split("```python")
        if len(parts) > 1:
            return parts[1].split("```")[0].strip()
    elif "```" in response_text:
        return response_text.split("```")[1].strip()
    return response_text.strip()

def measure_complexity(code: str) -> str:
    """
    Calcula una métrica simple de complejidad contando bucles y condicionales.
    """
    try:
        tree = ast.parse(code)
    except Exception:
        return "N/A"
    loops = sum(isinstance(node, (ast.For, ast.While)) for node in ast.walk(tree))
    conditionals = sum(isinstance(node, ast.If) for node in ast.walk(tree))
    return f"loops: {loops}, conditionals: {conditionals}"

def run_script_with_metrics(script_path: str, timeout: int = TIMEOUT_EXEC):
    """
    Ejecuta un script Python y mide su tiempo de ejecución y uso de memoria.
    Devuelve (resultado, tiempo_ejecución, memoria_MB).
    """
    start_time = time.time()
    process = psutil.Popen(["python", script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    max_memory = 0
    stdout, stderr = b"", b""
    while True:
        if process.poll() is not None:
            stdout, stderr = process.communicate()
            break
        if time.time() - start_time > timeout:
            process.kill()
            return "TIMEOUT", timeout, max_memory / (1024 * 1024)
        try:
            mem = process.memory_info().rss
            if mem > max_memory:
                max_memory = mem
        except psutil.NoSuchProcess:
            break
        time.sleep(0.1)
    exec_time = time.time() - start_time
    result = stdout.decode().strip() if stdout else stderr.decode().strip()
    return result, round(exec_time, 2), round(max_memory / (1024 * 1024), 2)

# -----------------------------------------------------------
#         CHECKPOINTS & RESUMPTION FUNCTIONS
# -----------------------------------------------------------
def is_nonsense(row) -> bool:
    """
    Función dummy para verificar si el resultado de una fila es inconsistente.
    """
    # Implementar la lógica necesaria si corresponde.
    return False

def load_checkpoint(csv_path: str):
    """
    Carga el CSV de checkpoint y devuelve un conjunto de claves (model, temperature, top_p, top_k, ProblemID)
    para los problemas procesados exitosamente (excluyendo fallos).
    """
    completed = set()
    if not os.path.exists(csv_path):
        return completed
    with open(csv_path, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["result"] not in ["FAIL_MAX_RETRIES", "ERROR"] and not is_nonsense(row):
                key = (row["model"], row["temperature"], row["top_p"], row["top_k"], row["ProblemID"])
                completed.add(key)
    return completed

def append_to_checkpoint(csv_path: str, fieldnames, rows: list):
    """
    Añade registros al CSV de checkpoint. Crea el archivo si no existe.
    """
    file_exists = os.path.exists(csv_path)
    with open(csv_path, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)

# -----------------------------------------------------------
#         PROBLEM PROCESSING WITH RETRIES
# -----------------------------------------------------------
def create_chat_session(llm_model: str, temperature: float, top_p: float, top_k: float):
    """
    Crea una sesión de chat con la configuración especificada.
    """
    generation_config = {
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "max_output_tokens": 8192,
        "response_mime_type": "text/plain",
    }
    model = genai.GenerativeModel(model_name=llm_model, generation_config=generation_config)
    return model.start_chat(history=[])

def call_model(chat_session, prompt_text: str):
    """
    Llama al modelo con un prompt y extrae el texto de respuesta y el uso de tokens (si está disponible).
    Devuelve (response_text, input_tokens, output_tokens).
    """
    try:
        response = chat_session.send_message(prompt_text)
        response_text = response.text
        in_tokens, out_tokens = None, None
        usage = getattr(response, "model_usage", None)
        if usage:
            in_tokens = usage.get("input_tokens", None)
            out_tokens = usage.get("output_tokens", None)
        return response_text, in_tokens, out_tokens
    except Exception as e:
        if "429" in str(e):
            print("[RATE LIMIT] Rotating API key due to 429 error.")
            rotate_api_key()
        raise e

def process_single_problem(problem, index, model, temp, top_p, top_k, output_dir):
    """
    Procesa un problema individual:
      - Rota la API key antes de iniciar.
      - Construye el prompt.
      - Llama al modelo para generar el código y extrae el uso de tokens.
      - Guarda el código generado en un archivo .py.
      - Ejecuta el código y mide tiempo de ejecución, memoria y complejidad.
    Devuelve un diccionario con todas las métricas recopiladas.
    """
    rotate_api_key()  # Rota la API key antes de procesar cada problema

    prompt_text = (
        "Solve the following problem in Python. The solution should implement a function that, "
        "given the specified input(s) in the problem, compares its output with the expected output and "
        "runs the tests. The function should print 'True' for each test passed and 'False' for each test "
        "failed, and finally print the number of correct tests over the total. "
        "Provide only executable Python code.\n\n" +
        f"{problem['Description']}"
    )
    chat_session = create_chat_session(model, temp, top_p, top_k)
    response_text, in_tokens, out_tokens = call_model(chat_session, prompt_text)
    python_code = extract_code(response_text)
    if is_nonsense_code(python_code):
        raise Exception("NONSENSE_CODE_DETECTED")
    output_file = Path(output_dir) / f"output_{index + 1}.py"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(python_code)
    print(f"[{model} | T={temp} | top_p={top_p} | top_k={top_k}] Code saved in: {output_file}")

    result, exec_time, mem_usage = run_script_with_metrics(str(output_file), TIMEOUT_EXEC)
    complexity = measure_complexity(python_code)
    true_count = result.count("True") if isinstance(result, str) else 0
    false_count = result.count("False") if isinstance(result, str) else 0

    return {
        "ID": str(index + 1),
        "ProblemID": problem['ID'],
        "model": model,
        "temperature": str(temp),
        "top_p": str(top_p),
        "top_k": str(top_k),
        "code": python_code,
        "result": result,
        "true_count": str(true_count),
        "false_count": str(false_count),
        "execution_time": str(exec_time),
        "memory_usage_MB": str(mem_usage),
        "algorithmic_complexity": complexity,
        "input_tokens": str(in_tokens) if in_tokens is not None else "N/A",
        "output_tokens": str(out_tokens) if out_tokens is not None else "N/A",
    }

def process_problem_with_retry(problem, index, model, temp, top_p, top_k, output_dir, max_attempts=5):
    """
    Intenta procesar un problema con reintentos y backoff progresivo.
    Solo se reintentan los errores de TIMEOUT.
    Si ocurre un error de ejecución (cualquier error que no sea TIMEOUT), se registra inmediatamente.
    Rota la API key en caso de error 429.
    """
    attempt = 0
    backoff_times = [15, 30, 60, 300]  # segundos
    current_backoff_index = 0
    while attempt < max_attempts:
        attempt += 1
        try:
            print(f"==> Processing Problem {problem['ID']} (Attempt {attempt}/{max_attempts})")
            start_local = time.time()
            result_dict = process_single_problem(problem, index, model, temp, top_p, top_k, output_dir)
            if result_dict["result"] == "TIMEOUT":
                if attempt < max_attempts:
                    print("[TIMEOUT] Received TIMEOUT. Retrying...")
                    wait_time = backoff_times[current_backoff_index] if current_backoff_index < len(backoff_times) else backoff_times[-1]
                    print(f"[RETRY] Waiting {wait_time} seconds before retrying.")
                    time.sleep(wait_time)
                    if current_backoff_index < len(backoff_times) - 1:
                        current_backoff_index += 1
                    continue
                else:
                    return result_dict
            else:
                return result_dict
        except Exception as e:
            if "TIMEOUT" in str(e):
                if attempt < max_attempts:
                    print(f"[ERROR] {e}. Retrying due to TIMEOUT error.")
                    wait_time = backoff_times[current_backoff_index] if current_backoff_index < len(backoff_times) else backoff_times[-1]
                    print(f"[RETRY] Waiting {wait_time} seconds before retrying.")
                    time.sleep(wait_time)
                    if current_backoff_index < len(backoff_times) - 1:
                        current_backoff_index += 1
                    continue
                else:
                    return {
                        "ID": str(index + 1),
                        "model": model,
                        "temperature": str(temp),
                        "top_p": str(top_p),
                        "top_k": str(top_k),
                        "code": "",
                        "result": "FAIL_MAX_RETRIES",
                        "true_count": "ERROR",
                        "false_count": "ERROR",
                        "execution_time": "ERROR",
                        "memory_usage_MB": "ERROR",
                        "algorithmic_complexity": "ERROR",
                        "input_tokens": "N/A",
                        "output_tokens": "N/A",
                    }
            else:
                return {
                    "ID": str(index + 1),
                    "model": model,
                    "temperature": str(temp),
                    "top_p": str(top_p),
                    "top_k": str(top_k),
                    "code": "",
                    "result": f"ERROR: {e}",
                    "true_count": "ERROR",
                    "false_count": "ERROR",
                    "execution_time": "ERROR",
                    "memory_usage_MB": "ERROR",
                    "algorithmic_complexity": "ERROR",
                    "input_tokens": "N/A",
                    "output_tokens": "N/A",
                }
    print(f"[FAIL] Problem {problem['ID']} failed after {max_attempts} attempts.")
    return {
        "ID": str(index + 1),
        "model": model,
        "temperature": str(temp),
        "top_p": str(top_p),
        "top_k": str(top_k),
        "code": "",
        "result": "FAIL_MAX_RETRIES",
        "true_count": "ERROR",
        "false_count": "ERROR",
        "execution_time": "ERROR",
        "memory_usage_MB": "ERROR",
        "algorithmic_complexity": "ERROR",
        "input_tokens": "N/A",
        "output_tokens": "N/A",
    }

def is_nonsense_code(code: str) -> bool:
    if not code or len(code.strip()) < 10:
        return True
    if all(c == code[0] for c in code.strip()):
        return True
    if "import" not in code and "def" not in code:
        return True
    return False

# -----------------------------------------------------------
#                 MAIN FUNCTION
# -----------------------------------------------------------
def main():
    """
    Función principal que:
      1. Configura el entorno y las API keys.
      2. Realiza el muestreo estratificado de problemas (objetivo 113) desde 'leetcode_problems.csv'.
      3. Guarda la asignación de la muestra en 'sample_allocation.csv'.
      4. Carga el checkpoint previo para reanudar la ejecución (reintentando solo errores TIMEOUT).
      5. Itera sobre cada combinación (modelos x conjuntos de parámetros) y procesa cada problema.
      6. Aplica backoff progresivo y reintentos (hasta 5 intentos) por problema.
      7. Detiene la ejecución si 3 problemas consecutivos fallan.
      8. Guarda los resultados incrementalmente en un CSV de checkpoint para reanudar.
      9. Reporta el progreso total (porcentaje completado).
    """
    try:
        setup_environment()
        # Realiza el muestreo estratificado (objetivo total = 113)
        try:
            if os.path.exists("sample_selected.csv"):
                print("Loading existing sample from sample_selected.csv")
                import pandas as pd
                sample_df = pd.read_csv("sample_selected.csv", encoding="utf-8")
                print(f"Loaded sample of {len(sample_df)} problems.")
            else:
                import pandas as pd
                sample_df = stratified_sampling(INPUT_FILENAME, TOTAL_SAMPLE_SIZE, VARIABILITY)
                print(f"Stratified sample selected: {len(sample_df)} problems.")
                sample_df.to_csv("sample_selected.csv", index=False, encoding="utf-8")
        except Exception as e:
            print(f"Error loading/creating sample: {e}")
            raise

        # Asegurarse de que el directorio de resultados exista
        Path(BASE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
        # Cargar el checkpoint para reanudar
        checkpoint_path = Path(BASE_RESULTS_DIR) / CHECKPOINT_CSV
        done_set = load_checkpoint(str(checkpoint_path))
        
        fieldnames = [
            "ID", "ProblemID", "model", "temperature", "top_p", "top_k",
            "code", "result", "true_count", "false_count",
            "execution_time", "memory_usage_MB", "algorithmic_complexity",
            "input_tokens", "output_tokens"
        ]
        
        consecutive_problem_fails = 0
        
        # Calcular total de tareas: combinaciones totales * número de problemas
        total_combinations = len(MODELS) * len(PARAM_SETS)
        total_tasks = total_combinations * len(sample_df)
        tasks_completed = 0
        
        # Iterar sobre cada combinación de modelo y conjunto de parámetros
        for model in MODELS:
            for params in PARAM_SETS:
                temp = params["temperature"]
                top_p = params["top_p"]
                top_k = params["top_k"]
                
                output_dir = Path(BASE_OUTPUT_DIR) / f"temperature-{temp}" / model / f"top_p-{top_p}_top_k-{top_k}"
                output_dir.mkdir(parents=True, exist_ok=True)
                
                print(f"\n--- Processing: Model={model}, Temp={temp}, top_p={top_p}, top_k={top_k} ---")
                
                for i, row in sample_df.iterrows():
                    combo_key = (model, str(temp), str(top_p), str(top_k), str(row["ID"]))
                    if combo_key in done_set:
                        print(f"[SKIP] Problem {row['ID']} with {combo_key} already processed successfully.")
                        tasks_completed += 1
                        progress = (tasks_completed / total_tasks) * 100
                        print(f"Progress: {progress:.2f}% complete.")
                        continue
                    
                    attempt_start = time.time()
                    result_dict = process_problem_with_retry(row, i, model, temp, top_p, top_k, str(output_dir))
                    append_to_checkpoint(str(checkpoint_path), fieldnames, [result_dict])
                    done_set.add(combo_key)
                    tasks_completed += 1
                    progress = (tasks_completed / total_tasks) * 100
                    print(f"Progress: {progress:.2f}% complete.")
                    
                    if result_dict["result"] in ["FAIL_MAX_RETRIES", "ERROR", "TIMEOUT"]:
                        consecutive_problem_fails += 1
                    else:
                        consecutive_problem_fails = 0
                    
                    elapsed = time.time() - attempt_start
                    time.sleep(max(0, 15 - elapsed))
                    
                    if consecutive_problem_fails >= 3:
                        print("[CRITICAL] 3 consecutive problems failed. Stopping execution.")
                        return
        
        print("\nExecution completed for all combinations and sampled problems.")
    except Exception as ex:
        print(f"[FATAL] Critical error: {ex}")

if __name__ == "__main__":
    main()


In [5]:
# Cargar variables de entorno y configurar el cliente de la API
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Modelos y parámetros
MODELS = ["o1-mini"]
TIMEOUT_EXEC = 300
INPUT_FILENAME = "sample_selected.csv"  # Usar el sample existente
OUTPUT_DIR = "outputs_openai"
RESULTS_FILE = "results_openai.csv"

def extract_code(response_text: str) -> str:
    if "```python" in response_text:
        return response_text.split("```python")[1].split("```")[0].strip()
    elif "```" in response_text:
        return response_text.split("```")[1].strip()
    return response_text.strip()

def measure_complexity(code: str) -> str:
    try:
        tree = ast.parse(code)
    except Exception:
        return "N/A"
    loops = sum(isinstance(node, (ast.For, ast.While)) for node in ast.walk(tree))
    conditionals = sum(isinstance(node, ast.If) for node in ast.walk(tree))
    return f"loops: {loops}, conditionals: {conditionals}"

def run_script(script_path: str, timeout: int):
    start_time = time.time()
    process = psutil.Popen(["python", script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    max_mem = 0
    stdout, stderr = b"", b""
    while True:
        if process.poll() is not None:
            stdout, stderr = process.communicate()
            break
        if time.time() - start_time > timeout:
            process.kill()
            return "TIMEOUT", timeout, max_mem / (1024 * 1024)
        try:
            mem = process.memory_info().rss
            if mem > max_mem:
                max_mem = mem
        except psutil.NoSuchProcess:
            break
        time.sleep(0.1)
    exec_time = time.time() - start_time
    result = stdout.decode().strip() if stdout else stderr.decode().strip()
    return result, round(exec_time, 2), round(max_mem / (1024 * 1024), 2)

def call_model(model: str, prompt: str):
    try:
        response = openai.ChatCompletion.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2048
        )
        text = response.choices[0].message["content"]
        usage = response["usage"]
        print(f"Model: {model}, Tokens Used: {usage['total_tokens']}")
        print(f"Response: {text}")
        return text, usage.get("prompt_tokens"), usage.get("completion_tokens")
    except openai.error.OpenAIError as e:
        print(f"Error: {e}")
        return None, None, None

def main():
    print("Starting OpenAI API processing...")
    df = pd.read_csv(INPUT_FILENAME, encoding="utf-8")  # Usar el sample existente
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    results = []
    print(f"Loaded {len(df)} problems from {INPUT_FILENAME}.")
    for model in MODELS:
        print(f"\n--- Processing: Model={model} ---")
        for i, row in df.iterrows():
            print(f"Processing Problem {row['ID']}...")
            prompt = (
                "Solve the following problem in Python. The solution should implement a function that, "
                "given the specified input(s) in the problem, compares its output with the expected output and "
                "runs the tests. The function should print 'True' for each test passed and 'False' for each test "
                "failed, and finally print the number of correct tests over the total. "
                "Provide only executable Python code.\n\n" + row["Description"]
            )

            try:
                response_text, in_tokens, out_tokens = call_model(model, prompt)
                code = extract_code(response_text)
                file_path = f"{OUTPUT_DIR}/output_{row['ID']}.py"
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(code)

                result, exec_time, mem_usage = run_script(file_path, TIMEOUT_EXEC)
                complexity = measure_complexity(code)

                true_count = result.count("True") if isinstance(result, str) else 0
                false_count = result.count("False") if isinstance(result, str) else 0

                results.append({
                    "ID": row["ID"],
                    "model": model,
                    "code": code,
                    "result": result,
                    "true_count": true_count,
                    "false_count": false_count,
                    "execution_time": exec_time,
                    "memory_usage_MB": mem_usage,
                    "algorithmic_complexity": complexity,
                    "input_tokens": in_tokens or "N/A",
                    "output_tokens": out_tokens or "N/A",
                    "full_response": response_text  # Guardar la respuesta completa
                })
                print(f"Processed Problem {row['ID']} successfully.")
            except Exception as e:
                print(f"Error processing Problem {row['ID']}: {e}")
                results.append({
                    "ID": row["ID"],
                    "model": model,
                    "code": "",
                    "result": str(e),
                    "true_count": "ERROR",
                    "false_count": "ERROR",
                    "execution_time": "ERROR",
                    "memory_usage_MB": "ERROR",
                    "algorithmic_complexity": "ERROR",
                    "input_tokens": "N/A",
                    "output_tokens": "N/A",
                    "full_response": response_text if 'response_text' in locals() else "N/A"
                })
            time.sleep(1)  # Pausa para evitar exceder los límites de tasa

    with open(RESULTS_FILE, "w", newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"Results saved to {RESULTS_FILE}")

if __name__ == "__main__":
    main()


In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import time
import csv
import ast
import psutil
from pathlib import Path
import pandas as pd
